## Question 4

In [13]:
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
import matplotlib.pyplot as plt

In [10]:
df = pd.read_csv('mushroom/agaricus-lepiota.data', header=None)

columns = [
    "class",
    "cap_shape",
    "cap_surface",
    "cap_color",
    "bruises",
    "odor",
    "gill_attachment",
    "gill_spacing",
    "gill_size",
    "gill_color",
    "stalk_shape",
    "stalk_root",
    "stalk_surface_above_ring",
    "stalk_surface_below_ring",
    "stalk_color_above_ring",
    "stalk_color_below_ring",
    "veil_type",
    "veil_color",
    "ring_number",
    "ring_type",
    "spore_print_color",
    "population",
    "habitat"
]

df.columns = columns
#print(df.info())


In [12]:
X = df.drop(columns='class')
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1
)

## Question 4.1

In [38]:
## Define Naive Bayes Model

class NaiveBayesClassifier:

    def fit(self, X, y, alpha=1):
        self.alpha = alpha
        self.classes_ = y.unique()
        self.prior_probs = {}
        self.cond_probs = {}
        n_total = len(y)

        for c in self.classes_:
            X_c = X[y == c]
            self.prior_probs[c] = len(X_c) / n_total


            self.cond_probs[c] = {}
            for col in X.columns:
                col_vals = X[col].values
                col_vals_c = X_c[col].values
                k = len(np.unique(col_vals))
                counts = defaultdict(int)
                for v in col_vals_c:
                    counts[v] += 1
                n_c = len(col_vals_c)
                
                self.cond_probs[c][col] = {
                    v: (counts[v] + alpha) / (n_c + alpha * k)
                    for v in np.unique(col_vals)
                }

        return self

    def _class_score(self, x, c):
        score = np.log(self.prior_probs[c])
        for col, val in x.items():
            prob = self.cond_probs[c][col].get(val, self.alpha / (self.alpha))
            score += np.log(prob)
        return score

    def predict(self, X):
        preds = []
        for _, row in X.iterrows():
            scores = {c: self._class_score(row, c) for c in self.classes_}
            preds.append(max(scores, key=scores.get))
        return np.array(preds)
    
    def print_probabilities(self):
        for col in list(self.cond_probs[self.classes_[0]].keys()):
            print(f"\n  Feature: {col}")
            for c in self.classes_:
                label = "Edible" if c == "e" else "Poisonous"
                print(f"    P({col} = x | Y={label}):")
                for val, prob in self.cond_probs[c][col].items():
                    print(f"      x={val}: {prob:.4f}")

    def predict_proba(self, X):
        all_probs = []
        for _, row in X.iterrows():
            scores = {c: self._class_score(row, c) for c in self.classes_}


            max_score = max(scores.values())
            exp_scores = {c: np.exp(s - max_score) for c, s in scores.items()}
            total = sum(exp_scores.values())
            probs = {c: exp_scores[c] / total for c in self.classes_}
            all_probs.append(probs)

        df = pd.DataFrame(all_probs, index=X.index)
        df.columns = ["Edible" if c == "e" else "Poisonous" for c in df.columns]
        return df



In [39]:
model = NaiveBayesClassifier()

model.fit(X_train, y_train)

In [36]:
print(model.print_probabilities())


  Feature: cap_shape
    P(cap_shape = x | Y=Poisonous):
      x=b: 0.0124
      x=c: 0.0010
      x=f: 0.3971
      x=k: 0.1597
      x=s: 0.0003
      x=x: 0.4294
    P(cap_shape = x | Y=Edible):
      x=b: 0.0930
      x=c: 0.0003
      x=f: 0.3851
      x=k: 0.0567
      x=s: 0.0063
      x=x: 0.4587

  Feature: cap_surface
    P(cap_surface = x | Y=Poisonous):
      x=f: 0.1918
      x=g: 0.0007
      x=s: 0.3579
      x=y: 0.4496
    P(cap_surface = x | Y=Edible):
      x=f: 0.3716
      x=g: 0.0003
      x=s: 0.2697
      x=y: 0.3584

  Feature: cap_color
    P(cap_color = x | Y=Poisonous):
      x=b: 0.0312
      x=c: 0.0027
      x=e: 0.2247
      x=g: 0.2099
      x=n: 0.2593
      x=p: 0.0237
      x=r: 0.0003
      x=u: 0.0003
      x=w: 0.0779
      x=y: 0.1698
    P(cap_color = x | Y=Edible):
      x=b: 0.0100
      x=c: 0.0091
      x=e: 0.1438
      x=g: 0.2455
      x=n: 0.3018
      x=p: 0.0131
      x=r: 0.0044
      x=u: 0.0038
      x=w: 0.1732
      x=y: 0.0954



## Question 4.2

In [43]:
predicted_probs = model.predict_proba(X_test)

for _, row in predicted_probs.iterrows():
    print(f"{_}: Poisonous: {row['Poisonous']:.4f}, Edible: {row['Edible']:.4f}, Actual {y_test[_]}")


1392: Poisonous: 0.0000, Edible: 1.0000, Actual e
4051: Poisonous: 0.2253, Edible: 0.7747, Actual p
3725: Poisonous: 0.7722, Edible: 0.2278, Actual p
7177: Poisonous: 1.0000, Edible: 0.0000, Actual p
103: Poisonous: 0.0000, Edible: 1.0000, Actual e
3371: Poisonous: 0.9315, Edible: 0.0685, Actual p
6738: Poisonous: 1.0000, Edible: 0.0000, Actual p
1525: Poisonous: 0.0000, Edible: 1.0000, Actual e
5838: Poisonous: 0.9963, Edible: 0.0037, Actual p
4299: Poisonous: 1.0000, Edible: 0.0000, Actual p
7549: Poisonous: 1.0000, Edible: 0.0000, Actual p
6082: Poisonous: 1.0000, Edible: 0.0000, Actual p
5189: Poisonous: 1.0000, Edible: 0.0000, Actual p
4258: Poisonous: 1.0000, Edible: 0.0000, Actual p
39: Poisonous: 0.0000, Edible: 1.0000, Actual e
7916: Poisonous: 0.0000, Edible: 1.0000, Actual e
6475: Poisonous: 1.0000, Edible: 0.0000, Actual p
2213: Poisonous: 0.0001, Edible: 0.9999, Actual e
3492: Poisonous: 1.0000, Edible: 0.0000, Actual p
783: Poisonous: 0.0000, Edible: 1.0000, Actual e
2079

## Question 4.3

In [46]:
y_pred = model.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label="p")
rec  = recall_score(y_test, y_pred, pos_label="p")
f1   = f1_score(y_test, y_pred, pos_label="p")


print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1 Score  : {f1:.4f}")

  Accuracy  : 0.9552
  Precision : 1.0000
  Recall    : 0.9100
  F1 Score  : 0.9529


## Question 4.4

In [49]:
## Train sklearn NB model

enc = OrdinalEncoder()
X_train_enc = enc.fit_transform(X_train)
X_test_enc  = enc.transform(X_test)


model_sklearn = CategoricalNB(alpha=1)
model_sklearn.fit(X_train_enc, y_train)
y_pred_sklearn = model_sklearn.predict(X_test_enc)

acc  = accuracy_score(y_test, y_pred_sklearn)
prec = precision_score(y_test, y_pred_sklearn, pos_label="p")
rec  = recall_score(y_test, y_pred_sklearn, pos_label="p")
f1   = f1_score(y_test, y_pred_sklearn, pos_label="p")


print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1 Score  : {f1:.4f}")

  Accuracy  : 0.9552
  Precision : 1.0000
  Recall    : 0.9100
  F1 Score  : 0.9529


In [51]:
## Get ROC-AUC score for both

proba_custom  = model.predict_proba(X_test)["Poisonous"].values
proba_sklearn = model_sklearn.predict_proba(X_test_enc)[:, 1]

y_binary = (y_test == "p").astype(int)

print(f"Custom model: {roc_auc_score(y_binary, proba_custom):.4f}")
print(f"Sklearn model: {roc_auc_score(y_binary, proba_sklearn):.4f}")

Custom model: 0.9982
Sklearn model: 0.9982


In [ ]:
custom_metrics = """
  Accuracy  : 0.9552
  Precision : 1.0000
  Recall    : 0.9100
  F1 Score  : 0.9529
  ROC-AUC   : 0.9982
"""

sklearn_metrics = """
  Accuracy  : 0.9552
  Precision : 1.0000
  Recall    : 0.9100
  F1 Score  : 0.9529
  ROC_AUC   : 0.9982
"""

## Analysis

Looking at the sklearn package metrics with our custom model, we see that the values are identical. This could be because of the all of the features are categorical, so when computing probabilities all both models are doing is counting, which will give them identical probabilities and they will produce identical results. With such even numbers and following the same mathematical formula for both models, they both produce the same results